# 06. Geospatial Clustering

Discover NYC neighborhood archetypes via k-Means.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np


In [ ]:
from src.utils.config import load_config
from src.utils.io import load_parquet
from src.models.clustering.kmeans import NeighborhoodKMeans

cfg = load_config()
df = load_parquet(f"{cfg['paths']['data_processed']}/properties_features.parquet")
print(df.shape)

## Find optimal k via elbow + silhouette

In [ ]:
km_probe = NeighborhoodKMeans(n_clusters=8, random_state=42)
scan = km_probe.find_optimal_k(df.sample(15000, random_state=42),
                              cfg['clustering']['features'])
from src.visualization.model_plots import plot_kmeans_optimal_k
plot_kmeans_optimal_k(scan, 'reports/figures/kmeans_optimal_k.png')

## Fit k-Means with chosen k

In [ ]:
km = NeighborhoodKMeans(n_clusters=8, random_state=42, n_init=10)
km.fit(df, cfg['clustering']['features'])
df['cluster'] = km.predict(df)
df['cluster_name'] = df['cluster'].map(km.cluster_names_)
print(km.cluster_names_)

## Cluster profiles

In [ ]:
km.cluster_profiles_

## Geo map (Folium)

In [ ]:
from src.visualization.geo_plots import plot_cluster_map, plot_valuation_map
m = plot_cluster_map(df, sample_size=4000, save_path='reports/figures/map_clusters.html')
m if m else 'install folium'

In [ ]:
m = plot_valuation_map(df, sample_size=4000, save_path='reports/figures/map_valuation.html')
m if m else 'install folium'

## Price HeatMap

Four interactive heatmaps showing how price, crime, walkability, and
undervalued-property concentration are distributed geographically.

In [ ]:
from src.visualization.geo_plots import plot_price_heatmap

# Price per sqft density
m_price = plot_price_heatmap(
    df,
    metric="price_per_sqft",
    sample_size=8000,
    save_path="reports/figures/heatmap_price.html",
)
print("Price heatmap: saved to reports/figures/heatmap_price.html")
m_price if m_price else "install folium"

In [ ]:
# Crime rate density
m_crime = plot_price_heatmap(
    df,
    metric="crime_rate_per_1k",
    sample_size=8000,
    save_path="reports/figures/heatmap_crime.html",
)
print("Crime heatmap: saved to reports/figures/heatmap_crime.html")
m_crime if m_crime else "install folium"

In [ ]:
# Walk score density
m_walk = plot_price_heatmap(
    df,
    metric="walk_score",
    sample_size=8000,
    save_path="reports/figures/heatmap_walkability.html",
)
print("Walk score heatmap: saved to reports/figures/heatmap_walkability.html")
m_walk if m_walk else "install folium"

In [ ]:
# Undervalued property hotspots
m_uv = plot_price_heatmap(
    df,
    metric="undervalued",
    sample_size=8000,
    save_path="reports/figures/heatmap_undervalued.html",
)
print("Undervalued hotspot map: saved to reports/figures/heatmap_undervalued.html")
m_uv if m_uv else "install folium"

## Hierarchical comparison

In [ ]:
from src.models.clustering.hierarchical import NeighborhoodHierarchical
from src.visualization.model_plots import plot_dendrogram
hc = NeighborhoodHierarchical(n_clusters=8)
hc.fit(df.sample(2000, random_state=42), cfg['clustering']['features'])
linkage = hc.get_dendrogram_data(df.sample(500, random_state=42))
plot_dendrogram(linkage, 'reports/figures/dendrogram.png')

## Save

In [ ]:
from src.utils.io import save_parquet
import joblib
from pathlib import Path
save_parquet(df, f"{cfg['paths']['data_processed']}/properties_clustered.parquet")
km.save(Path(cfg['paths']['models']) / 'clustering_kmeans.pkl')